
## JPmart - Fake Data Generator

Generates synthetic e-commerce data for the fictional JPmart lakehouse project, with ~8% intentional dirtiness (nulls, duplicates, inconsistent types, malformed dates, invalid emails) so the Silver layer has real cleaning work to do.

Entities generated directly into the Bronze volume as CSV, matching how each one actually behaves in a real e-commerce system:
- **products**    (~200 rows) — single file, small reference data.
- **customers**   (~1,000 initial rows) — arrives in batches: new
  signups plus occasional profile updates (email change, loyalty
  status) mixed into the same file, like a real upsert/CDC source.
- **orders**       (~5,000 initial rows) — arrives in batches: new
  orders plus status changes on existing ones (pending -> shipped ->
  delivered), same upsert idea.
- **order_items**  (~1-4 items per order) — append-only, one file per
  arrival batch of orders; items are never updated once written.
- **web_events**   (~20,000 rows) — append-only, split into multiple
  batch files (like hourly clickstream exports).

`01_ingest_bronze` picks up `customers`/`orders` incrementally with Auto Loader + `MERGE INTO` (upsert), and `order_items`/`web_events` incrementally with Auto Loader as plain append — instead of reloading everything on every run.

Output: `/Volumes/jpmart/bronze/raw_files/`

## Install dependencies
Faker is not preinstalled on the Databricks Runtime, so it needs to be
installed on the cluster before importing it.

In [0]:
%pip install faker

In [0]:
dbutils.library.restartPython()

## Configuration

In [0]:
import csv
import os
import random
import uuid
from datetime import datetime, timedelta

from faker import Faker

SEED = 42
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

N_CUSTOMERS = 1_000
N_PRODUCTS = 200
N_ORDERS = 5_000
N_WEB_EVENTS = 20_000
N_WEB_EVENT_BATCHES = 10  # simulate 10 arrival batches instead of one dump

DIRTY_RATE = 0.08

OUTPUT_DIR = "/Volumes/jpmart/bronze/raw_files"
CUSTOMERS_DIR = f"{OUTPUT_DIR}/customers"
ORDERS_DIR = f"{OUTPUT_DIR}/orders"
ORDER_ITEMS_DIR = f"{OUTPUT_DIR}/order_items"
WEB_EVENTS_DIR = f"{OUTPUT_DIR}/web_events"
for _dir in (OUTPUT_DIR, CUSTOMERS_DIR, ORDERS_DIR, ORDER_ITEMS_DIR, WEB_EVENTS_DIR):
    os.makedirs(_dir, exist_ok=True)

CATEGORIES = [
    "Electronics", "Home & Kitchen", "Clothing", "Books",
    "Sports & Outdoors", "Beauty", "Toys & Games", "Grocery",
]

ORDER_STATUS_PROGRESSION = {"pending": "shipped", "shipped": "delivered"}

US_STATES = [
    "VA", "MD", "DC", "NY", "CA", "TX", "FL", "IL", "PA", "OH",
]

## Intentional "dirtiness" helpers



In [0]:
def maybe_dirty(value, dirty_value, rate=DIRTY_RATE):
    """Returns dirty_value with probability `rate`, otherwise returns value."""
    r = random.random()
    return dirty_value if r < rate else value


def dirty_email(email):
    """Introduces invalid emails: missing @, malformed domain, or null."""
    r = random.random()
    if r < DIRTY_RATE * 0.4:
        return None
    if r < DIRTY_RATE * 0.7:
        return email.replace("@", "_at_")  # no valid @
    if r < DIRTY_RATE:
        return email.upper() + "  "  # uppercase + extra spaces
    return email


def dirty_date(date_obj, fmt_pool):
    """Returns the date in a random format (sometimes inconsistent)."""
    r = random.random()
    if r < DIRTY_RATE:
        fmt = random.choice(fmt_pool)
        return date_obj.strftime(fmt)
    return date_obj.strftime("%Y-%m-%d")


DATE_FORMATS_DIRTY = ["%d/%m/%Y", "%m-%d-%Y", "%Y/%m/%d", "%B %d, %Y"]


def dirty_price(price):
    """Sometimes turns the price into a string with a currency symbol,
    makes it negative (simulated capture error), or drops it to null."""
    r = random.random()
    if r < DIRTY_RATE * 0.3:
        return f"${price}"  # string instead of float
    if r < DIRTY_RATE * 0.5:
        return -abs(price)  # negative price (capture error)
    if r < DIRTY_RATE:
        return None
    return price

## Entity generators

In [0]:
def generate_customers(n):
    rows = []
    for i in range(1, n + 1):
        first = fake.first_name()
        last = fake.last_name()
        email = f"{first.lower()}.{last.lower()}{random.randint(1, 999)}@{fake.free_email_domain()}"
        signup_date = fake.date_between(start_date="-3y", end_date="today")

        row = {
            "customer_id": f"JPM-CUST-{i:06d}",
            "first_name": first,
            "last_name": last,
            "email": dirty_email(email),
            "signup_date": dirty_date(signup_date, DATE_FORMATS_DIRTY),
            "state": maybe_dirty(random.choice(US_STATES), random.choice(US_STATES).lower()),
            "loyalty_member": maybe_dirty(random.choice([True, False]), None),
        }
        rows.append(row)

    # Inject exact duplicates (~2% of rows)
    n_dupes = int(n * 0.02)
    rows += random.sample(rows, n_dupes)
    random.shuffle(rows)
    return rows


def generate_products(n):
    rows = []
    for i in range(1, n + 1):
        base_price = round(random.uniform(5, 500), 2)
        row = {
            "product_id": f"JPM-PROD-{i:06d}",
            "product_name": fake.catch_phrase(),
            "category": maybe_dirty(random.choice(CATEGORIES), None),
            "unit_price": dirty_price(base_price),
            "supplier": fake.company(),
            "active": maybe_dirty(True, "yes"),  # bool vs string inconsistency
        }
        rows.append(row)
    return rows


def generate_orders(n, customer_ids):
    rows = []
    for i in range(1, n + 1):
        order_date = fake.date_time_between(start_date="-2y", end_date="now")
        row = {
            "order_id": f"JPM-ORD-{i:08d}",
            "customer_id": maybe_dirty(random.choice(customer_ids), None),  # orphan FK
            "order_date": dirty_date(order_date, DATE_FORMATS_DIRTY),
            "status": maybe_dirty(
                random.choice(ORDER_STATUSES), random.choice(ORDER_STATUSES).upper()
            ),
        }
        rows.append(row)

    n_dupes = int(n * 0.015)
    rows += random.sample(rows, n_dupes)
    random.shuffle(rows)
    return rows


def generate_order_items(orders, product_ids, start_counter=1):
    """start_counter lets later arrival batches keep generating unique order_item_ids instead of restarting from 1 and colliding with
    previously-written items."""
    rows = []
    item_counter = start_counter
    for order in orders:
        n_items = random.randint(1, 4)
        for _ in range(n_items):
            qty = random.randint(1, 5)
            unit_price = round(random.uniform(5, 500), 2)
            row = {
                "order_item_id": f"JPM-ORDITEM-{item_counter:08d}",
                "order_id": order["order_id"],
                "product_id": maybe_dirty(random.choice(product_ids), None),
                "quantity": maybe_dirty(qty, -qty),  # negative quantity error
                "unit_price": dirty_price(unit_price),
            }
            rows.append(row)
            item_counter += 1

    # Inject exact duplicates (~2%). order_items stays append-only (no MERGE), so - unlike customers/orders below - these duplicates flow
    # through to Bronze untouched, preserving a real "dedupe it in Silver" exercise even after customers/orders move to upsert ingestion.
    n_dupes = int(len(rows) * 0.02)
    if n_dupes:
        rows += random.sample(rows, n_dupes)
        random.shuffle(rows)
    return rows

EVENT_TYPES = ["page_view", "product_view", "add_to_cart", "remove_from_cart", "checkout_start", "purchase"]


def generate_web_events(n, customer_ids, product_ids):
    rows = []
    for i in range(1, n + 1):
        event_time = fake.date_time_between(start_date="-6M", end_date="now")
        row = {
            "event_id": f"JPM-EVT-{i:08d}",
            "customer_id": maybe_dirty(random.choice(customer_ids + [None] * 50), None),  # anonymous sessions
            "product_id": maybe_dirty(random.choice(product_ids), None),
            "event_type": random.choice(EVENT_TYPES),
            "event_timestamp": dirty_date(event_time, DATE_FORMATS_DIRTY),
            "session_id": str(uuid.uuid4()),
        }
        rows.append(row)
    return rows

## Write to the Bronze volume

In [0]:
def write_csv(rows, filename):
    if not rows:
        return
    path = os.path.join(OUTPUT_DIR, filename)
    fieldnames = list(rows[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"  -> {filename}: {len(rows):,} rows")


def write_web_events_batches(rows, n_batches, prefix="web_events_batch"):
    """Splits web_events rows into several files under WEB_EVENTS_DIR, as if they had arrived in separate exports instead of a single bulk load."""
    if not rows:
        return
    batch_size = math.ceil(len(rows) / n_batches)
    fieldnames = list(rows[0].keys())
    for i in range(n_batches):
        batch = rows[i * batch_size : (i + 1) * batch_size]
        if not batch:
            continue
        filename = f"{prefix}_{i + 1:03d}.csv"
        path = os.path.join(WEB_EVENTS_DIR, filename)
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(batch)
        print(f"  -> web_events/{filename}: {len(batch):,} rows")


def write_batch_file(rows, directory, prefix):
    """Writes a single batch file into `directory`"""
    if not rows:
        return
    next_num = len(os.listdir(directory)) + 1
    filename = f"{prefix}_{next_num:03d}.csv"
    path = os.path.join(directory, filename)
    fieldnames = list(rows[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return filename


def reset_dir(directory):
    """Clears out files left over from a previous run of the generation cell below"""
    for f in os.listdir(directory):
        os.remove(os.path.join(directory, f))

## Run generation

In [0]:

print("Generating fake data for JPmart...\n")

for _dir in (CUSTOMERS_DIR, ORDERS_DIR, ORDER_ITEMS_DIR, WEB_EVENTS_DIR):
    reset_dir(_dir)

print("Products...")
products = generate_products(N_PRODUCTS)
write_csv(products, "products.csv")
product_ids = [p["product_id"] for p in products]

print("Customers (initial batch)...")
customers = generate_customers(N_CUSTOMERS)
customers_file = write_batch_file(customers, CUSTOMERS_DIR, "customers_batch")
print(f"  -> customers/{customers_file}: {len(customers):,} rows")
customer_ids = [c["customer_id"] for c in customers]

print("Orders (initial batch)...")
orders = generate_orders(N_ORDERS, customer_ids)
orders_file = write_batch_file(orders, ORDERS_DIR, "orders_batch")
print(f"  -> orders/{orders_file}: {len(orders):,} rows")

print("Order items (initial batch)...")
order_items = generate_order_items(orders, product_ids)
order_items_file = write_batch_file(order_items, ORDER_ITEMS_DIR, "order_items_batch")
print(f"  -> order_items/{order_items_file}: {len(order_items):,} rows")

print("Web events (split into arrival batches)...")
web_events = generate_web_events(N_WEB_EVENTS, customer_ids, product_ids)
write_web_events_batches(web_events, N_WEB_EVENT_BATCHES)

print(f"\nDone. Files written to: {OUTPUT_DIR}")

## Simulate new arrivals (for the streaming/upsert demo)

Run these cells any time after the main generation above to drop new files into the volume, as if new data had just arrived:
- `simulate_new_web_events_batch` — pure append (new events only).
- `simulate_new_customers_batch` — mixes brand-new signups **and** profile updates on existing customers into the same file, like a real upsert/CDC source would.
- `simulate_new_orders_and_items_batch` — same idea for orders (new orders + status progressions on existing ones), plus a matching append-only order_items file for the brand-new orders.

Re-run the matching Auto Loader cell in `01_ingest_bronze` afterward — only the new/changed rows should move, not a full reload.

In [0]:
def simulate_new_web_events_batch(n_rows=2000):
    filename = write_batch_file(
        generate_web_events(n_rows, customer_ids, product_ids),
        WEB_EVENTS_DIR, "web_events_batch",
    )
    print(f"New arrival simulated -> web_events/{filename}: {n_rows:,} rows")


def simulate_new_customers_batch(n_new=50, n_updates=30):
    """One file containing brand-new signups plus a sample of existing customers re-emitted with a changed attribute (email/loyalty status) -
    a real upsert source usually mixes inserts and updates together too."""
    start_id = len(customers) + 1
    new_rows = []
    for offset in range(n_new):
        idx = start_id + offset
        first = fake.first_name()
        last = fake.last_name()
        email = f"{first.lower()}.{last.lower()}{random.randint(1, 999)}@{fake.free_email_domain()}"
        signup_date = fake.date_between(start_date="-3y", end_date="today")
        new_rows.append({
            "customer_id": f"JPM-CUST-{idx:06d}",
            "first_name": first,
            "last_name": last,
            "email": dirty_email(email),
            "signup_date": dirty_date(signup_date, DATE_FORMATS_DIRTY),
            "state": maybe_dirty(random.choice(US_STATES), random.choice(US_STATES).lower()),
            "loyalty_member": maybe_dirty(random.choice([True, False]), None),
        })

    existing_sample = random.sample(customers, min(n_updates, len(customers)))
    updated_rows = []
    for row in existing_sample:
        updated = dict(row)
        updated["email"] = (
            f"{row['first_name'].lower()}.{row['last_name'].lower()}"
            f".new{random.randint(1, 999)}@{fake.free_email_domain()}"
        )
        updated["loyalty_member"] = not bool(row.get("loyalty_member"))
        updated_rows.append(updated)

    customers.extend(new_rows)
    customer_ids.extend(c["customer_id"] for c in new_rows)
    by_id = {c["customer_id"]: c for c in customers}
    for updated in updated_rows:
        by_id[updated["customer_id"]].update(updated)

    batch = new_rows + updated_rows
    random.shuffle(batch)
    filename = write_batch_file(batch, CUSTOMERS_DIR, "customers_batch")
    print(f"New customers batch -> customers/{filename}: {len(new_rows)} new, {len(updated_rows)} updated")


def simulate_new_orders_and_items_batch(n_new_orders=200, n_status_updates=150):
    """One orders file (new orders + status progressions on existing ones) plus a matching append-only order_items file for the new orders only -
    order_items are never updated, only ever added."""
    start_id = len(orders) + 1
    new_orders = []
    for offset in range(n_new_orders):
        idx = start_id + offset
        order_date = fake.date_time_between(start_date="-2y", end_date="now")
        new_orders.append({
            "order_id": f"JPM-ORD-{idx:08d}",
            "customer_id": maybe_dirty(random.choice(customer_ids), None),
            "order_date": dirty_date(order_date, DATE_FORMATS_DIRTY),
            "status": maybe_dirty(
                random.choice(ORDER_STATUSES), random.choice(ORDER_STATUSES).upper()
            ),
        })

    progressable = [o for o in orders if o["status"].strip().lower() in ORDER_STATUS_PROGRESSION]
    existing_sample = random.sample(progressable, min(n_status_updates, len(progressable)))
    updated_orders = []
    for row in existing_sample:
        updated = dict(row)
        updated["status"] = ORDER_STATUS_PROGRESSION[row["status"].strip().lower()]
        updated_orders.append(updated)

    orders.extend(new_orders)
    by_id = {o["order_id"]: o for o in orders}
    for updated in updated_orders:
        by_id[updated["order_id"]]["status"] = updated["status"]

    orders_batch = new_orders + updated_orders
    random.shuffle(orders_batch)
    orders_filename = write_batch_file(orders_batch, ORDERS_DIR, "orders_batch")
    print(f"New orders batch -> orders/{orders_filename}: {len(new_orders)} new, {len(updated_orders)} status updates")

    # order_items only for the brand-new orders - append-only, so no updates here
    new_items = generate_order_items(new_orders, product_ids, start_counter=len(order_items) + 1)
    order_items.extend(new_items)
    items_filename = write_batch_file(new_items, ORDER_ITEMS_DIR, "order_items_batch")
    print(f"New order_items batch -> order_items/{items_filename}: {len(new_items):,} rows")


# Uncomment to actually simulate new arrivals:
# simulate_new_web_events_batch()
# simulate_new_customers_batch()
# simulate_new_orders_and_items_batch()

## Validation
Quick sanity check: read each CSV back with Spark, confirm row counts, and check that dirty values (nulls, malformed emails/dates/prices) are actually showing up at roughly the expected ~8% rate.

In [0]:
df = spark.read.option("header", "true").csv(f"{OUTPUT_DIR}/products.csv")
print(f"products.csv: {df.count():,} rows")
df.display()

for name, directory in [
    ("customers", CUSTOMERS_DIR),
    ("orders", ORDERS_DIR),
    ("order_items", ORDER_ITEMS_DIR),
    ("web_events", WEB_EVENTS_DIR),
]:
    df = spark.read.option("header", "true").csv(directory)
    print(f"{name}/ ({len(os.listdir(directory))} files): {df.count():,} rows")
    df.display()


# Example: check the null rate on customers.email to confirm dirtiness landed
customers_df = spark.read.option("header", "true").csv(CUSTOMERS_DIR)
total = customers_df.count()
null_emails = customers_df.filter(customers_df.email.isNull()).count()
print(f"Null emails: {null_emails} / {total} ({null_emails / total:.1%})")